# OhioT1DM Dataset - Glucose & Insulin Prediction Models

This notebook trains machine learning models for:
1. **Glucose Prediction**: Predicting future blood glucose levels
2. **Insulin Recommendation**: Predicting optimal insulin dosage

**Dataset**: OhioT1DM - Type 1 Diabetes dataset with CGM, insulin, and meal data

**Models**: LSTM (for glucose time-series), XGBoost (for insulin recommendation)

## 1. Import Required Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Machine Learning
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Deep Learning (LSTM for time series)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, GRU
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# XGBoost for insulin prediction
import xgboost as xgb
from xgboost import XGBRegressor

# Save models
import joblib
import pickle

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"XGBoost version: {xgb.__version__}")

## 2. Load the OhioT1DM Dataset

The OhioT1DM dataset contains:
- **CGM (Continuous Glucose Monitor)**: Blood glucose readings every 5 minutes
- **Insulin**: Basal and bolus insulin doses
- **Meal/Carbs**: Carbohydrate intake from meals
- **Exercise**: Physical activity data
- **Timestamps**: All data is timestamped

Dataset structure: Each patient has separate XML files for different data types.

In [ ]:
# Define path to OhioT1DM dataset
# Update this path to where you downloaded the dataset
DATA_PATH = './OhioT1DM_data/'  # Change to your dataset path

# Patient IDs in OhioT1DM dataset
# Training set: 559, 563, 570, 575, 588, 591
# Testing set: 540, 544, 552, 567, 584, 596
PATIENT_IDS = ['559', '563', '570', '575', '588', '591']

def load_patient_xml_data(patient_id, data_type):
    """
    Load XML data for a specific patient and data type
    
    Parameters:
    - patient_id: Patient identifier (e.g., '559')
    - data_type: Type of data ('glucose', 'insulin', 'meal', 'exercise')
    
    Returns:
    - DataFrame with the parsed data
    """
    file_path = f"{DATA_PATH}{patient_id}-ws-training.xml"
    
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        
        data_list = []
        
        if data_type == 'glucose':
            for event in root.findall('.//glucose_level/event'):
                timestamp = event.get('ts')
                value = event.get('value')
                data_list.append({
                    'timestamp': pd.to_datetime(timestamp),
                    'glucose': float(value)
                })
        
        elif data_type == 'insulin':
            # Basal insulin
            for event in root.findall('.//basal/event'):
                timestamp = event.get('ts')
                value = event.get('value')
                data_list.append({
                    'timestamp': pd.to_datetime(timestamp),
                    'basal_insulin': float(value),
                    'bolus_insulin': 0.0
                })
            
            # Bolus insulin
            for event in root.findall('.//bolus/event'):
                timestamp = event.get('ts')
                dose_normal = event.get('dose_normal', '0')
                dose_square = event.get('dose_square', '0')
                total_dose = float(dose_normal) + float(dose_square)
                data_list.append({
                    'timestamp': pd.to_datetime(timestamp),
                    'basal_insulin': 0.0,
                    'bolus_insulin': total_dose
                })
        
        elif data_type == 'meal':
            for event in root.findall('.//meal/event'):
                timestamp = event.get('ts')
                carbs = event.get('carbs', '0')
                data_list.append({
                    'timestamp': pd.to_datetime(timestamp),
                    'carbs': float(carbs)
                })
        
        elif data_type == 'exercise':
            for event in root.findall('.//exercise/event'):
                timestamp = event.get('ts')
                duration = event.get('duration', '0')
                intensity = event.get('intensity', 'medium')
                data_list.append({
                    'timestamp': pd.to_datetime(timestamp),
                    'exercise_duration': float(duration),
                    'exercise_intensity': intensity
                })
        
        return pd.DataFrame(data_list)
    
    except Exception as e:
        print(f"Error loading {data_type} data for patient {patient_id}: {e}")
        return pd.DataFrame()

print("Data loading functions defined successfully!")

In [ ]:
# Load data for all patients
all_patients_data = []

for patient_id in PATIENT_IDS:
    print(f"\nLoading data for patient {patient_id}...")
    
    # Load different data types
    glucose_df = load_patient_xml_data(patient_id, 'glucose')
    insulin_df = load_patient_xml_data(patient_id, 'insulin')
    meal_df = load_patient_xml_data(patient_id, 'meal')
    exercise_df = load_patient_xml_data(patient_id, 'exercise')
    
    # Store patient data
    patient_data = {
        'patient_id': patient_id,
        'glucose': glucose_df,
        'insulin': insulin_df,
        'meal': meal_df,
        'exercise': exercise_df
    }
    
    all_patients_data.append(patient_data)
    
    print(f"  Glucose records: {len(glucose_df)}")
    print(f"  Insulin records: {len(insulin_df)}")
    print(f"  Meal records: {len(meal_df)}")
    print(f"  Exercise records: {len(exercise_df)}")

print("\n✓ Data loaded successfully for all patients!")

## 3. Merge and Preprocess Data

Combine all data sources into a single time-aligned DataFrame with 5-minute intervals.

In [ ]:
def merge_patient_data(patient_data):
    """
    Merge all data sources for a patient into single DataFrame
    """
    glucose_df = patient_data['glucose'].copy()
    insulin_df = patient_data['insulin'].copy()
    meal_df = patient_data['meal'].copy()
    
    if glucose_df.empty:
        return pd.DataFrame()
    
    # Set timestamp as index for glucose (our primary data)
    glucose_df.set_index('timestamp', inplace=True)
    glucose_df = glucose_df.sort_index()
    
    # Create 5-minute interval index
    start_time = glucose_df.index.min()
    end_time = glucose_df.index.max()
    time_index = pd.date_range(start=start_time, end=end_time, freq='5min')
    
    # Reindex glucose data
    df = glucose_df.reindex(time_index)
    df['glucose'] = df['glucose'].interpolate(method='linear')
    
    # Merge insulin data (forward fill for basal, sum for bolus in time windows)
    if not insulin_df.empty:
        insulin_df.set_index('timestamp', inplace=True)
        insulin_df = insulin_df.sort_index()
        
        # Resample insulin to 5-min intervals
        insulin_resampled = insulin_df.resample('5min').agg({
            'basal_insulin': 'mean',
            'bolus_insulin': 'sum'
        })
        
        df = df.join(insulin_resampled, how='left')
        df['basal_insulin'] = df['basal_insulin'].fillna(method='ffill').fillna(0)
        df['bolus_insulin'] = df['bolus_insulin'].fillna(0)
    else:
        df['basal_insulin'] = 0
        df['bolus_insulin'] = 0
    
    # Merge meal data (sum carbs in 5-min windows)
    if not meal_df.empty:
        meal_df.set_index('timestamp', inplace=True)
        meal_df = meal_df.sort_index()
        
        meal_resampled = meal_df.resample('5min').agg({
            'carbs': 'sum'
        })
        
        df = df.join(meal_resampled, how='left')
        df['carbs'] = df['carbs'].fillna(0)
    else:
        df['carbs'] = 0
    
    # Reset index to have timestamp as column
    df.reset_index(inplace=True)
    df.rename(columns={'index': 'timestamp'}, inplace=True)
    
    # Add patient ID
    df['patient_id'] = patient_data['patient_id']
    
    return df

# Merge data for all patients
merged_data_list = []

for patient_data in all_patients_data:
    patient_id = patient_data['patient_id']
    print(f"Merging data for patient {patient_id}...")
    
    merged_df = merge_patient_data(patient_data)
    
    if not merged_df.empty:
        merged_data_list.append(merged_df)
        print(f"  Total records: {len(merged_df)}")

# Combine all patients data
df_combined = pd.concat(merged_data_list, ignore_index=True)

print(f"\n✓ Combined dataset shape: {df_combined.shape}")
print(f"✓ Date range: {df_combined['timestamp'].min()} to {df_combined['timestamp'].max()}")
print(f"\nFirst few rows:")
df_combined.head()

## 4. Exploratory Data Analysis

In [ ]:
# Basic statistics
print("Dataset Statistics:")
print(df_combined.describe())

print("\n\nData Info:")
print(df_combined.info())

print("\n\nMissing Values:")
print(df_combined.isnull().sum())

# Visualize data distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Glucose distribution
axes[0, 0].hist(df_combined['glucose'].dropna(), bins=50, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Blood Glucose Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Glucose (mg/dL)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df_combined['glucose'].mean(), color='red', linestyle='--', label=f'Mean: {df_combined["glucose"].mean():.1f}')
axes[0, 0].legend()

# Insulin distribution (total)
df_combined['total_insulin'] = df_combined['basal_insulin'] + df_combined['bolus_insulin']
axes[0, 1].hist(df_combined[df_combined['total_insulin'] > 0]['total_insulin'], bins=50, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Insulin Dose Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Insulin (units)')
axes[0, 1].set_ylabel('Frequency')

# Carbs distribution
axes[1, 0].hist(df_combined[df_combined['carbs'] > 0]['carbs'], bins=50, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Carbohydrate Intake Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Carbs (grams)')
axes[1, 0].set_ylabel('Frequency')

# Time series plot (sample patient)
sample_patient = df_combined[df_combined['patient_id'] == PATIENT_IDS[0]].iloc[:1000]
axes[1, 1].plot(sample_patient['timestamp'], sample_patient['glucose'], color='blue', linewidth=1)
axes[1, 1].set_title(f'Glucose Time Series (Patient {PATIENT_IDS[0]} - First 1000 points)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Glucose (mg/dL)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Feature Engineering

Create temporal, lag, and rolling window features for prediction.

In [ ]:
def create_features(df):
    """
    Create comprehensive features for glucose and insulin prediction
    """
    df = df.copy()
    
    # Temporal features
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_morning'] = ((df['hour'] >= 6) & (df['hour'] < 12)).astype(int)
    df['is_afternoon'] = ((df['hour'] >= 12) & (df['hour'] < 18)).astype(int)
    df['is_evening'] = ((df['hour'] >= 18) & (df['hour'] < 22)).astype(int)
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] < 6)).astype(int)
    
    # Cyclical encoding for hour (sine/cosine)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # Group by patient for patient-specific features
    for patient_id in df['patient_id'].unique():
        patient_mask = df['patient_id'] == patient_id
        
        # Lag features for glucose (previous values)
        for lag in [1, 2, 3, 6, 12]:  # 5, 10, 15, 30, 60 minutes
            df.loc[patient_mask, f'glucose_lag_{lag}'] = df.loc[patient_mask, 'glucose'].shift(lag)
        
        # Rolling window statistics for glucose
        for window in [6, 12, 24]:  # 30 min, 1 hour, 2 hours
            df.loc[patient_mask, f'glucose_mean_{window}'] = df.loc[patient_mask, 'glucose'].rolling(window=window, min_periods=1).mean()
            df.loc[patient_mask, f'glucose_std_{window}'] = df.loc[patient_mask, 'glucose'].rolling(window=window, min_periods=1).std()
            df.loc[patient_mask, f'glucose_min_{window}'] = df.loc[patient_mask, 'glucose'].rolling(window=window, min_periods=1).min()
            df.loc[patient_mask, f'glucose_max_{window}'] = df.loc[patient_mask, 'glucose'].rolling(window=window, min_periods=1).max()
        
        # Glucose rate of change (trend)
        df.loc[patient_mask, 'glucose_diff_1'] = df.loc[patient_mask, 'glucose'].diff(1)
        df.loc[patient_mask, 'glucose_diff_2'] = df.loc[patient_mask, 'glucose'].diff(2)
        
        # Cumulative insulin and carbs in recent window
        for window in [12, 24, 48]:  # 1 hour, 2 hours, 4 hours
            df.loc[patient_mask, f'insulin_sum_{window}'] = (
                df.loc[patient_mask, 'basal_insulin'] + df.loc[patient_mask, 'bolus_insulin']
            ).rolling(window=window, min_periods=1).sum()
            
            df.loc[patient_mask, f'carbs_sum_{window}'] = df.loc[patient_mask, 'carbs'].rolling(window=window, min_periods=1).sum()
    
    # Total insulin
    df['total_insulin'] = df['basal_insulin'] + df['bolus_insulin']
    
    return df

# Create features
print("Creating features...")
df_features = create_features(df_combined)

# Remove NaN values from lag features
df_features = df_features.dropna()

print(f"✓ Features created! New shape: {df_features.shape}")
print(f"✓ Feature columns: {df_features.shape[1]}")
print("\nFeature columns:")
print(df_features.columns.tolist())

## 6. Prepare Data for Glucose Prediction

**Goal**: Predict glucose level 30 minutes into the future (6 timesteps ahead)

**Input Features**: 
- Current and historical glucose values
- Recent insulin doses
- Recent carb intake
- Time of day

**Target**: Glucose value at t+6 (30 minutes ahead)

In [ ]:
# Create target variable (glucose 30 minutes ahead)
PREDICTION_HORIZON = 6  # 6 * 5min = 30 minutes

df_glucose = df_features.copy()

# Create target by shifting glucose backwards (future value)
for patient_id in df_glucose['patient_id'].unique():
    patient_mask = df_glucose['patient_id'] == patient_id
    df_glucose.loc[patient_mask, 'glucose_target'] = df_glucose.loc[patient_mask, 'glucose'].shift(-PREDICTION_HORIZON)

# Remove rows with NaN target
df_glucose = df_glucose.dropna(subset=['glucose_target'])

# Select features for glucose prediction
glucose_features = [
    # Lag features
    'glucose_lag_1', 'glucose_lag_2', 'glucose_lag_3', 'glucose_lag_6', 'glucose_lag_12',
    # Rolling statistics
    'glucose_mean_6', 'glucose_std_6', 'glucose_min_6', 'glucose_max_6',
    'glucose_mean_12', 'glucose_std_12', 'glucose_min_12', 'glucose_max_12',
    'glucose_mean_24', 'glucose_std_24',
    # Trend features
    'glucose_diff_1', 'glucose_diff_2',
    # Insulin and carbs
    'insulin_sum_12', 'insulin_sum_24', 'insulin_sum_48',
    'carbs_sum_12', 'carbs_sum_24', 'carbs_sum_48',
    # Temporal features
    'hour_sin', 'hour_cos', 'is_morning', 'is_afternoon', 'is_evening', 'is_night'
]

X_glucose = df_glucose[glucose_features].values
y_glucose = df_glucose['glucose_target'].values

print(f"Glucose prediction dataset:")
print(f"  X shape: {X_glucose.shape}")
print(f"  y shape: {y_glucose.shape}")
print(f"  Features: {len(glucose_features)}")

# Split data (temporal split - last 20% for testing)
split_idx = int(len(X_glucose) * 0.8)
X_train_glucose = X_glucose[:split_idx]
X_test_glucose = X_glucose[split_idx:]
y_train_glucose = y_glucose[:split_idx]
y_test_glucose = y_glucose[split_idx:]

print(f"\nTrain set: {X_train_glucose.shape}")
print(f"Test set: {X_test_glucose.shape}")

# Normalize features
scaler_glucose = StandardScaler()
X_train_glucose_scaled = scaler_glucose.fit_transform(X_train_glucose)
X_test_glucose_scaled = scaler_glucose.transform(X_test_glucose)

print("\n✓ Data prepared for glucose prediction!")

## 7. Build LSTM Model for Glucose Prediction

LSTM (Long Short-Term Memory) is ideal for time-series prediction as it can capture temporal dependencies.

In [ ]:
# Reshape data for LSTM (samples, timesteps, features)
# We'll use a single timestep for simplicity, but you can extend this to use sequences
X_train_lstm = X_train_glucose_scaled.reshape((X_train_glucose_scaled.shape[0], 1, X_train_glucose_scaled.shape[1]))
X_test_lstm = X_test_glucose_scaled.reshape((X_test_glucose_scaled.shape[0], 1, X_test_glucose_scaled.shape[1]))

print(f"LSTM input shape: {X_train_lstm.shape}")

# Build LSTM model
def build_glucose_lstm_model(input_shape):
    model = Sequential([
        LSTM(128, activation='relu', return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(64, activation='relu', return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.1),
        Dense(16, activation='relu'),
        Dense(1)  # Output: predicted glucose value
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Create model
model_glucose_lstm = build_glucose_lstm_model((X_train_lstm.shape[1], X_train_lstm.shape[2]))

print("\nModel architecture:")
model_glucose_lstm.summary()

In [ ]:
# Train LSTM model
print("Training LSTM model for glucose prediction...")

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
checkpoint = ModelCheckpoint('models/glucose_lstm_best.h5', monitor='val_loss', save_best_only=True)

history_glucose = model_glucose_lstm.fit(
    X_train_lstm, y_train_glucose,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print("\n✓ LSTM model trained successfully!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history_glucose.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history_glucose.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('LSTM Training Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history_glucose.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history_glucose.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_title('LSTM Training MAE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluate Glucose Prediction Model

In [ ]:
# Make predictions
y_pred_glucose = model_glucose_lstm.predict(X_test_lstm).flatten()

# Calculate metrics
rmse_glucose = np.sqrt(mean_squared_error(y_test_glucose, y_pred_glucose))
mae_glucose = mean_absolute_error(y_test_glucose, y_pred_glucose)
r2_glucose = r2_score(y_test_glucose, y_pred_glucose)

print("=" * 60)
print("GLUCOSE PREDICTION MODEL PERFORMANCE (30 min ahead)")
print("=" * 60)
print(f"RMSE: {rmse_glucose:.2f} mg/dL")
print(f"MAE:  {mae_glucose:.2f} mg/dL")
print(f"R² Score: {r2_glucose:.4f}")
print("=" * 60)

# Visualize predictions
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Actual vs Predicted (scatter plot)
axes[0].scatter(y_test_glucose, y_pred_glucose, alpha=0.5, s=10)
axes[0].plot([y_test_glucose.min(), y_test_glucose.max()], 
             [y_test_glucose.min(), y_test_glucose.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_title('Glucose Prediction: Actual vs Predicted', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Actual Glucose (mg/dL)')
axes[0].set_ylabel('Predicted Glucose (mg/dL)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Time series comparison (first 500 samples)
sample_size = min(500, len(y_test_glucose))
axes[1].plot(y_test_glucose[:sample_size], label='Actual', linewidth=2, alpha=0.7)
axes[1].plot(y_pred_glucose[:sample_size], label='Predicted', linewidth=2, alpha=0.7)
axes[1].set_title('Glucose Prediction Time Series (First 500 test samples)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time Index')
axes[1].set_ylabel('Glucose (mg/dL)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Error distribution
errors = y_test_glucose - y_pred_glucose
plt.figure(figsize=(10, 6))
plt.hist(errors, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
plt.title('Glucose Prediction Error Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Prediction Error (mg/dL)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. Prepare Data for Insulin Recommendation

**Goal**: Predict optimal insulin dosage based on current state

**Input Features**:
- Current glucose level and trend
- Recent carb intake
- Time of day
- Recent insulin doses

**Target**: Total insulin dose (basal + bolus)

In [ ]:
# Filter data where insulin was actually administered (non-zero)
df_insulin = df_features[df_features['total_insulin'] > 0].copy()

print(f"Insulin dosing records: {len(df_insulin)}")

# Select features for insulin prediction
insulin_features = [
    # Current glucose state
    'glucose', 'glucose_lag_1', 'glucose_lag_2', 'glucose_lag_3',
    # Glucose statistics
    'glucose_mean_12', 'glucose_std_12',
    # Glucose trend
    'glucose_diff_1', 'glucose_diff_2',
    # Recent carbs
    'carbs', 'carbs_sum_12', 'carbs_sum_24',
    # Recent insulin (for context)
    'basal_insulin', 'bolus_insulin',
    # Temporal
    'hour_sin', 'hour_cos', 'is_morning', 'is_afternoon', 'is_evening'
]

X_insulin = df_insulin[insulin_features].values
y_insulin = df_insulin['total_insulin'].values

print(f"\nInsulin prediction dataset:")
print(f"  X shape: {X_insulin.shape}")
print(f"  y shape: {y_insulin.shape}")
print(f"  Features: {len(insulin_features)}")

# Split data (temporal split)
split_idx = int(len(X_insulin) * 0.8)
X_train_insulin = X_insulin[:split_idx]
X_test_insulin = X_insulin[split_idx:]
y_train_insulin = y_insulin[:split_idx]
y_test_insulin = y_insulin[split_idx:]

print(f"\nTrain set: {X_train_insulin.shape}")
print(f"Test set: {X_test_insulin.shape}")

# Normalize features
scaler_insulin = StandardScaler()
X_train_insulin_scaled = scaler_insulin.fit_transform(X_train_insulin)
X_test_insulin_scaled = scaler_insulin.transform(X_test_insulin)

print("\n✓ Data prepared for insulin prediction!")

## 10. Build XGBoost Model for Insulin Prediction

XGBoost is excellent for regression tasks with complex non-linear relationships.

In [ ]:
# Build XGBoost model for insulin prediction
print("Training XGBoost model for insulin recommendation...")

model_insulin_xgb = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    random_state=42,
    verbosity=1
)

# Train model
model_insulin_xgb.fit(
    X_train_insulin_scaled, y_train_insulin,
    eval_set=[(X_test_insulin_scaled, y_test_insulin)],
    early_stopping_rounds=20,
    verbose=10
)

print("\n✓ XGBoost model trained successfully!")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': insulin_features,
    'importance': model_insulin_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features for Insulin Prediction:")
print(feature_importance.head(10))

# Plot feature importance
plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'].head(15), feature_importance['importance'].head(15))
plt.xlabel('Importance')
plt.title('Top 15 Features for Insulin Prediction', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 11. Evaluate Insulin Prediction Model

In [ ]:
# Make predictions
y_pred_insulin = model_insulin_xgb.predict(X_test_insulin_scaled)

# Calculate metrics
rmse_insulin = np.sqrt(mean_squared_error(y_test_insulin, y_pred_insulin))
mae_insulin = mean_absolute_error(y_test_insulin, y_pred_insulin)
r2_insulin = r2_score(y_test_insulin, y_pred_insulin)

print("=" * 60)
print("INSULIN PREDICTION MODEL PERFORMANCE")
print("=" * 60)
print(f"RMSE: {rmse_insulin:.2f} units")
print(f"MAE:  {mae_insulin:.2f} units")
print(f"R² Score: {r2_insulin:.4f}")
print("=" * 60)

# Visualize predictions
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Actual vs Predicted (scatter plot)
axes[0].scatter(y_test_insulin, y_pred_insulin, alpha=0.5, s=20)
axes[0].plot([y_test_insulin.min(), y_test_insulin.max()], 
             [y_test_insulin.min(), y_test_insulin.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_title('Insulin Prediction: Actual vs Predicted', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Actual Insulin (units)')
axes[0].set_ylabel('Predicted Insulin (units)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Time series comparison (first 200 samples)
sample_size = min(200, len(y_test_insulin))
x_axis = np.arange(sample_size)
axes[1].bar(x_axis, y_test_insulin[:sample_size], alpha=0.6, label='Actual', width=0.4)
axes[1].bar(x_axis + 0.4, y_pred_insulin[:sample_size], alpha=0.6, label='Predicted', width=0.4)
axes[1].set_title('Insulin Prediction Comparison (First 200 test samples)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Insulin (units)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Error distribution
errors_insulin = y_test_insulin - y_pred_insulin
plt.figure(figsize=(10, 6))
plt.hist(errors_insulin, bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
plt.title('Insulin Prediction Error Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Prediction Error (units)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 12. Save Models for Production

Save trained models and scalers for deployment in the ML service.

In [ ]:
import os

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save LSTM model for glucose prediction
model_glucose_lstm.save('models/glucose_lstm_model.h5')
print("✓ Saved: models/glucose_lstm_model.h5")

# Save glucose scaler
joblib.dump(scaler_glucose, 'models/glucose_scaler.pkl')
print("✓ Saved: models/glucose_scaler.pkl")

# Save glucose feature names
with open('models/glucose_features.pkl', 'wb') as f:
    pickle.dump(glucose_features, f)
print("✓ Saved: models/glucose_features.pkl")

# Save XGBoost model for insulin prediction
joblib.dump(model_insulin_xgb, 'models/insulin_xgb_model.pkl')
print("✓ Saved: models/insulin_xgb_model.pkl")

# Save insulin scaler
joblib.dump(scaler_insulin, 'models/insulin_scaler.pkl')
print("✓ Saved: models/insulin_scaler.pkl")

# Save insulin feature names
with open('models/insulin_features.pkl', 'wb') as f:
    pickle.dump(insulin_features, f)
print("✓ Saved: models/insulin_features.pkl")

# Save model metadata
metadata = {
    'glucose_model': {
        'type': 'LSTM',
        'prediction_horizon': 30,  # minutes
        'prediction_horizon_steps': PREDICTION_HORIZON,
        'input_features': len(glucose_features),
        'rmse': rmse_glucose,
        'mae': mae_glucose,
        'r2_score': r2_glucose,
        'feature_names': glucose_features
    },
    'insulin_model': {
        'type': 'XGBoost',
        'input_features': len(insulin_features),
        'rmse': rmse_insulin,
        'mae': mae_insulin,
        'r2_score': r2_insulin,
        'feature_names': insulin_features
    },
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': 'OhioT1DM',
    'num_patients': len(PATIENT_IDS)
}

with open('models/model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("✓ Saved: models/model_metadata.pkl")

print("\n" + "=" * 60)
print("ALL MODELS SAVED SUCCESSFULLY!")
print("=" * 60)
print(f"\nGlucose Model: LSTM (RMSE: {rmse_glucose:.2f} mg/dL, R²: {r2_glucose:.4f})")
print(f"Insulin Model: XGBoost (RMSE: {rmse_insulin:.2f} units, R²: {r2_insulin:.4f})")
print("\nModels are ready for deployment in the ML service!")

## 13. Example: Making Predictions

Demonstrate how to use the trained models for new predictions.

In [ ]:
# Example: Predict glucose 30 minutes ahead
print("EXAMPLE: Glucose Prediction")
print("=" * 60)

# Take a sample from test set
sample_idx = 100
sample_features = X_test_glucose_scaled[sample_idx:sample_idx+1]
actual_glucose = y_test_glucose[sample_idx]

# Reshape for LSTM
sample_lstm = sample_features.reshape((1, 1, sample_features.shape[1]))

# Predict
predicted_glucose = model_glucose_lstm.predict(sample_lstm, verbose=0)[0][0]

print(f"Current glucose (from lag features): {X_test_glucose[sample_idx][0]:.1f} mg/dL")
print(f"Predicted glucose (30 min ahead): {predicted_glucose:.1f} mg/dL")
print(f"Actual glucose (30 min ahead): {actual_glucose:.1f} mg/dL")
print(f"Prediction error: {abs(predicted_glucose - actual_glucose):.1f} mg/dL")

print("\n" + "=" * 60)
print("EXAMPLE: Insulin Recommendation")
print("=" * 60)

# Take a sample from insulin test set
sample_idx_insulin = 50
sample_features_insulin = X_test_insulin_scaled[sample_idx_insulin:sample_idx_insulin+1]
actual_insulin = y_test_insulin[sample_idx_insulin]

# Predict
predicted_insulin = model_insulin_xgb.predict(sample_features_insulin)[0]

print(f"Current glucose: {X_test_insulin[sample_idx_insulin][0]:.1f} mg/dL")
print(f"Recent carbs: {X_test_insulin[sample_idx_insulin][9]:.1f} g")
print(f"Predicted insulin dose: {predicted_insulin:.1f} units")
print(f"Actual insulin dose: {actual_insulin:.1f} units")
print(f"Prediction error: {abs(predicted_insulin - actual_insulin):.1f} units")

print("\n✓ Prediction examples completed!")

## Summary & Next Steps

### Model Performance Summary

**Glucose Prediction Model (LSTM)**
- Predicts blood glucose 30 minutes into the future
- Input: Historical glucose, insulin, carbs, time features
- Output: Future glucose level (mg/dL)

**Insulin Recommendation Model (XGBoost)**
- Recommends insulin dosage based on current state
- Input: Current glucose, carbs, trend, time features
- Output: Insulin dose (units)

### Integration with ML Service

The trained models are saved in the `models/` directory and can be loaded in your FastAPI service:

```python
# In glucose_predictor.py
from tensorflow.keras.models import load_model
import joblib

glucose_model = load_model('models/glucose_lstm_model.h5')
glucose_scaler = joblib.load('models/glucose_scaler.pkl')

# In insulin_advisor.py
insulin_model = joblib.load('models/insulin_xgb_model.pkl')
insulin_scaler = joblib.load('models/insulin_scaler.pkl')
```

### Required Features for Production

When making predictions, ensure you provide these features:
- **Glucose**: Recent glucose readings (lag 1, 2, 3, 6, 12)
- **Rolling Statistics**: Mean, std, min, max over different windows
- **Insulin/Carbs**: Cumulative sums over recent windows
- **Temporal**: Hour sin/cos, time of day flags

### Next Steps

1. **Model Improvement**: 
   - Add more patients from OhioT1DM test set
   - Try GRU or Bidirectional LSTM
   - Implement ensemble methods

2. **Feature Engineering**:
   - Add exercise intensity features
   - Include sleep quality indicators
   - Add stress/illness flags

3. **Production Deployment**:
   - Create prediction API endpoints
   - Add input validation
   - Implement logging and monitoring

4. **Safety Measures**:
   - Add prediction confidence intervals
   - Implement alerts for dangerous predictions
   - Require doctor approval for insulin recommendations